### Tools

#### Models can request to call tools that perform tasks such as fetching data from a database, searching de web, or running code. Tools are pairing of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response) 
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using function. Use get_weather.', 'tool_calls': [{'id': 'fc_80ff7474-c9fd-4348-8f1f-62c14c7e689b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 127, 'total_tokens': 169, 'completion_time': 0.088213213, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.005426785, 'prompt_tokens_details': None, 'queue_time': 0.27281668, 'total_time': 0.093639998}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a4be3b0989', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a16169-df38-7449-b179-7f48e3cf7747-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_80ff7474-c9fd-4348-8f1f-62c14c7e689b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'inpu

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        result = get_weather.invoke(tool_call["args"])
        print(result)

It's sunny in Boston


### Tool Excution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute  the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72F and sunny"


It's sunny in Boston right now.


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in Boston. We have function get_weather. We should call it.', 'tool_calls': [{'id': 'fc_f1a66894-bf92-4904-afd4-cad10e0cb74c', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 126, 'total_tokens': 173, 'completion_time': 0.099085865, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.004726653, 'prompt_tokens_details': None, 'queue_time': 0.207277494, 'total_time': 0.103812518}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1617d-5354-720e-a61d-edddd32b0bec-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_f1a668